## EDA
1. Structural integrity
2. Missing values
3. Descriptive statistics + sanity bounds
4. Infinite values
5. Target distribution — class balance
6. Multicollinearity — correlation matrix
7. Stationarity check
8. Outlier detection on returns/volume

In [6]:
import pandas as pd
import numpy as np

### 1. Structural integrity

In [ ]:
df = pd.read_csv('datasets/historical/BTC_USDT_1h.csv', parse_dates=['timestamp'])
df = df.set_index('timestamp')

print(df.shape)
print(df.index.min(), df.index.max())
expected = pd.date_range(df.index.min(), df.index.max(), freq='h')
missing_timestamps = expected.difference(df.index)
print(f"Missing hours: {len(missing_timestamps)}")
print(missing_timestamps[:20])

(77684, 102)
2017-08-17 04:00:00+00:00 2026-07-03 07:00:00+00:00
Missing hours: 128
DatetimeIndex(['2017-09-06 17:00:00+00:00', '2017-09-06 18:00:00+00:00',
               '2017-09-06 19:00:00+00:00', '2017-09-06 20:00:00+00:00',
               '2017-09-06 21:00:00+00:00', '2017-09-06 22:00:00+00:00',
               '2018-01-04 04:00:00+00:00', '2018-02-08 01:00:00+00:00',
               '2018-02-08 02:00:00+00:00', '2018-02-08 03:00:00+00:00',
               '2018-02-08 04:00:00+00:00', '2018-02-08 05:00:00+00:00',
               '2018-02-08 06:00:00+00:00', '2018-02-08 07:00:00+00:00',
               '2018-02-08 08:00:00+00:00', '2018-02-08 09:00:00+00:00',
               '2018-02-08 10:00:00+00:00', '2018-02-08 11:00:00+00:00',
               '2018-02-08 12:00:00+00:00', '2018-02-08 13:00:00+00:00'],
              dtype='datetime64[us, UTC]', freq=None)


In [3]:
missing_rows = df[df.isnull().any(axis=1)]
print(missing_rows)

                               open      high       low     close      volume  \
timestamp                                                                       
2017-08-17 04:00:00+00:00   4261.48   4313.62   4261.32   4308.83   47.181009   
2017-08-17 05:00:00+00:00   4308.83   4328.69   4291.37   4315.32   23.234916   
2017-08-17 06:00:00+00:00   4330.29   4345.45   4309.37   4324.35    7.229691   
2017-08-17 07:00:00+00:00   4316.62   4349.99   4287.41   4349.99    4.443249   
2017-08-17 08:00:00+00:00   4333.32   4377.85   4333.32   4360.69    0.972807   
...                             ...       ...       ...       ...         ...   
2026-07-03 03:00:00+00:00  61440.01  61575.15  61324.00  61434.00  592.377010   
2026-07-03 04:00:00+00:00  61434.00  61524.01  61332.76  61448.00  597.687440   
2026-07-03 05:00:00+00:00  61448.00  61808.00  61430.00  61700.49  669.420270   
2026-07-03 06:00:00+00:00  61700.49  61850.00  61612.00  61710.01  821.565500   
2026-07-03 07:00:00+00:00  6

### 2. Missing values

In [4]:
na_counts = df.isna().sum()
print(na_counts[na_counts > 0].sort_values(ascending=False))

# Specifically check rolling-window warmup columns
for col in ['high_100', 'low_100', 'ema_200', 'senkou_a', 'senkou_b']:
    print(col, df[col].isna().sum(), df[col].first_valid_index())

vol_regime_ratio     119
pct_from_low          99
pct_from_high         99
low_100               99
high_100              99
senkou_b              77
senkou_a              51
sma_50                49
dist_resist           49
dist_support          49
resist_50             49
support_50            49
sr_width              49
ret_48                48
kijun                 25
target_dir_24h        24
target_ret_24h        24
ret_24                24
vs_vwap               23
vwap_24               23
vol_regime_20         20
bb_width              19
vol_stddev            19
vol_ma20              19
vol_ratio             19
sma_20                19
obv_ma20              19
bb_mid                19
bb_pct                19
bb_upper              19
bb_lower              19
cci                   19
stoch_d               15
trend_strength        14
stoch_k               13
willr                 13
ret_12                12
target_dir_12h        12
target_ret_12h        12
ema200_slope          10


### 3. Descriptive statistics + sanity bounds


In [7]:
# RSI
print("\n===== RSI_14 =====")
print(f"NaN values: {df['rsi_14'].isna().sum()}")

invalid = df[df['rsi_14'].notna() & ~df['rsi_14'].between(0, 100)]

if invalid.empty:
    print("✓ All non-NaN RSI values are valid.")
else:
    print(f"✗ {len(invalid)} invalid RSI values found.")
    print(invalid[['rsi_14']])

# Bollinger %
print("\n===== BB_PCT =====")
print(f"NaN values: {df['bb_pct'].isna().sum()}")

invalid = df[df['bb_pct'].notna() & ~df['bb_pct'].between(-5, 5)]

if invalid.empty:
    print("✓ All non-NaN bb_pct values are valid.")
else:
    print(f"✗ {len(invalid)} invalid bb_pct values found.")
    print(invalid[['bb_pct']])

# Infinite values
print("\n===== Infinite Values =====")

numeric = df.select_dtypes(include=[np.number])

for col in numeric.columns:
    inf_rows = numeric[np.isinf(numeric[col])]
    if not inf_rows.empty:
        print(f"\nColumn: {col}")
        print(inf_rows[[col]])

if not np.isinf(numeric).any().any():
    print("✓ No infinite values found.")


===== RSI_14 =====
NaN values: 1
✓ All non-NaN RSI values are valid.

===== BB_PCT =====
NaN values: 19
✓ All non-NaN bb_pct values are valid.

===== Infinite Values =====
✓ No infinite values found.


### 4. Infinite values 

In [8]:
num_cols = df.select_dtypes(include=[np.number]).columns
inf_mask = np.isinf(df[num_cols])
print(inf_mask.sum()[inf_mask.sum() > 0])

Series([], dtype: int64)


### 5. Target distribution


In [9]:
for col in ['target_dir_1h', 'target_dir_4h', 'target_dir_12h', 'target_dir_24h']:
    print(col)
    print(df[col].value_counts(normalize=True, dropna=False))
    print()

target_dir_1h
target_dir_1h
1.0    0.731631
2.0    0.137210
0.0    0.131147
NaN    0.000013
Name: proportion, dtype: float64

target_dir_4h
target_dir_4h
1.0    0.498751
2.0    0.259075
0.0    0.242122
NaN    0.000051
Name: proportion, dtype: float64

target_dir_12h
target_dir_12h
2.0    0.357281
0.0    0.329965
1.0    0.312600
NaN    0.000154
Name: proportion, dtype: float64

target_dir_24h
target_dir_24h
2.0    0.411243
0.0    0.374852
1.0    0.213596
NaN    0.000309
Name: proportion, dtype: float64

